# 5. Dynamic circuits and classical control flow

**목표.** Qiskit v2.x 자격증 시험에 필요한 최소한의 dynamic circuit 문법을 익힌다.  
핵심은 mid-circuit measurement, classical feedforward, `if_test`, `reset`, 그리고 control-flow block의 의미이다.


## 요약 대상 source notebook

| Source notebook | 중요하게 볼 내용 | 가볍게 볼 내용 |
|---|---|---|
| `1_Construct_Dynamic_Circuits.ipynb` | mid-circuit measurement, `if_test`, classical control, reset | 긴 응용 예제 |
| `2_4_Circuit_Mechanics.ipynb` | `Instruction`, `ControlFlowOp`, circuit data 구조 | 일반 circuit mechanics와 중복되는 부분 |

**핵심 아이디어.** Dynamic circuit은 회로 실행 중 qubit을 측정하고, 그 classical 결과에 따라 뒤의 quantum operation을 바꾸는 circuit이다.


In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit import QuantumRegister, ClassicalRegister


## 1. Mid-circuit measurement와 `if_test`

Dynamic circuit의 가장 기본 형태는 중간에 측정하고, 그 결과가 특정 값이면 gate를 적용하는 것이다.


In [ ]:
q = QuantumRegister(1, "q")
c = ClassicalRegister(1, "c")
qc = QuantumCircuit(q, c)

qc.h(q[0])
qc.measure(q[0], c[0])

with qc.if_test((c[0], 1)):
    qc.x(q[0])

qc.measure(q[0], c[0])
qc.draw("text")


## 2. `if` / `else` block

`if_test`를 context manager로 저장하면 `else` block도 만들 수 있다.


In [ ]:
q = QuantumRegister(2, "q")
c = ClassicalRegister(2, "c")
qc = QuantumCircuit(q, c)

qc.h(q[0])
qc.measure(q[0], c[0])

with qc.if_test((c[0], 1)) as else_:
    qc.h(q[1])
with else_:
    qc.x(q[1])

qc.measure(q[1], c[1])
qc.draw("text")


## 3. Classical register 조건

조건은 하나의 classical bit뿐 아니라 classical register 전체 값에도 걸 수 있다.


In [ ]:
q = QuantumRegister(3, "q")
c = ClassicalRegister(3, "c")
qc = QuantumCircuit(q, c)

qc.h(q[0])
qc.h(q[1])
qc.measure(q[0], c[0])
qc.measure(q[1], c[1])

# classical register 값이 01이면 q[2]에 X를 적용한다.
with qc.if_test((c, 0b001)):
    qc.x(q[2])

qc.measure(q[2], c[2])
qc.draw("text")


## 4. `reset`과 qubit 재사용

`reset`은 qubit을 다시 \(|0\rangle\) 상태로 만든다.  
Dynamic circuit에서는 측정한 qubit을 reset한 뒤 다시 사용할 수 있다.


In [ ]:
qc = QuantumCircuit(1, 1)

qc.h(0)
qc.measure(0, 0)
qc.reset(0)
qc.x(0)
qc.measure(0, 0)

qc.draw("text")


## 5. Control-flow operation

`if_test`, `for_loop`, `while_loop`, `switch` 같은 block은 circuit 안의 classical control-flow를 표현한다.  
다만 실제 backend에서 지원되는 control-flow 기능은 backend별로 다를 수 있다.


In [ ]:
qc = QuantumCircuit(1)

with qc.for_loop(range(3)):
    qc.x(0)

qc.draw("text")


## 요약

| 목적 | 표현 | 의미 |
|---|---|---|
| 중간 측정 | `qc.measure(q, c)` | 실행 중 qubit 정보를 classical bit에 저장 |
| 조건부 실행 | `with qc.if_test((cbit, 1)):` | classical bit 값이 1이면 block 실행 |
| else branch | `with qc.if_test(...) as else_:` | if가 실행되지 않을 때의 branch 생성 |
| register 조건 | `with qc.if_test((creg, value)):` | classical register 전체 값을 조건으로 사용 |
| qubit 초기화 | `qc.reset(q)` | qubit을 \(|0\rangle\)로 되돌림 |
| 반복 block | `with qc.for_loop(...):` | 반복 control-flow operation 생성 |
| 핵심 차이 | static vs dynamic | dynamic은 측정 결과가 뒤 operation에 영향을 줌 |


## 자격증 시험 스타일 연습문제

각 문제에서 a), b), c), d) 중 하나를 고르시오.

1. Dynamic circuit의 핵심 특징은 무엇인가?

a) circuit을 항상 measurement 없이 실행한다.  
b) 실행 중 측정 결과에 따라 뒤의 operation을 바꿀 수 있다.  
c) 모든 gate를 classical gate로 바꾼다.  
d) transpilation을 사용하지 않는다.

2. 중간 측정 결과가 1일 때만 `x(0)`을 적용하려면 어떤 표현이 가장 적절한가?

a) `with qc.if_test((c[0], 1)):`  
b) `with qc.for_loop((c[0], 1)):`  
c) `qc.assign_parameters({c[0]: 1})`  
d) `qc.decompose(c[0])`

3. `if_test`에서 `else` block을 만들 때 필요한 것은 무엇인가?

a) `qc.measure_all()`의 반환값  
b) `qc.if_test(...) as else_`로 받은 context manager  
c) `QuantumRegister`의 이름  
d) backend name

4. `reset`의 의미로 가장 적절한 것은 무엇인가?

a) qubit을 \(|0\rangle\) 상태로 초기화한다.  
b) classical register를 삭제한다.  
c) circuit depth를 0으로 만든다.  
d) 모든 parameter를 bind한다.

5. `with qc.if_test((creg, 0b001)):`의 의미는 무엇인가?

a) qubit register의 크기를 1로 만든다.  
b) classical register 값이 주어진 bitstring일 때 block을 실행한다.  
c) feature map을 만든다.  
d) 모든 gate를 measurement로 바꾼다.

6. 다음 중 일반적으로 unitary gate가 아닌 instruction은 무엇인가?

a) `rx`  
b) `cx`  
c) `measure`  
d) `h`

7. `for_loop`, `while_loop`, `if_test`는 어떤 종류의 circuit operation과 관련이 깊은가?

a) `ControlFlowOp`  
b) `ParameterVector`  
c) `SparsePauliOp`  
d) `EquivalenceLibrary`

8. Dynamic circuit을 실제 backend에서 실행할 때 특히 확인해야 할 것은 무엇인가?

a) backend가 해당 control-flow 기능을 지원하는지  
b) notebook filename이 짧은지  
c) Python 변수가 모두 대문자인지  
d) circuit에 parameter가 전혀 없는지


## 간단한 정답 해설

1. **b)** dynamic circuit은 mid-circuit measurement와 classical feedforward를 사용할 수 있다.  
2. **a)** `if_test((cbit, 1))`는 classical bit 값이 1일 때 block을 실행한다.  
3. **b)** `as else_`로 받은 context manager를 사용해 else branch를 만든다.  
4. **a)** `reset`은 qubit을 \(|0\rangle\) 상태로 되돌린다.  
5. **b)** classical register 전체 값이 조건값과 같을 때 block을 실행한다.  
6. **c)** measurement는 non-unitary instruction이다.  
7. **a)** 조건문과 반복문은 control-flow operation이다.  
8. **a)** 실제 hardware 지원 여부는 backend별로 다를 수 있다.
